# TSIS — Lab 4

## Monitoring agentic AI systems

An agent that works in a notebook can still fail in production: it may call the wrong tool, loop, become slow, consume too many tokens, or expose sensitive data. **Tracing** records the execution as a tree of runs (agent, model calls, tools, retrievers), while **monitoring** turns those traces into operational signals and alerts.

In this lab we instrument the same small LangChain agent with **LangSmith** (easy to use, but proprietary), **LangFuse** (an open-source alternative), and an **OpenTelemetry (OTel)** exporter. The goal is not to pick a universal winner: it is to understand what data is collected, where it is sent, and how the choice affects debugging and production operations.

### Learning objectives

By the end of this lab you should be able to:

- distinguish logs, metrics, traces, and evaluations for AI agents;
- trace an agent and inspect nested model/tool calls;
- attach useful metadata such as project, environment, user/session IDs, and tags;
- compare a hosted proprietary platform with an open-source observability platform;
- explain how OTel and an OTel Collector can route spans to more than one backend;
- make an informed decision about privacy, retention, sampling, cost, and incident response.


## Setup

Run the next cell once. The LangFuse package is needed for the second half of the lab; the OpenTelemetry packages are included for the annotated example near the end. Then fill in the LLM gateway key supplied by the teaching team. Never commit keys or paste real user data into a hosted observability service.


In [ ]:
%pip install --quiet --upgrade langchain langchain-openai langgraph langfuse opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http

In [ ]:
API_KEY = "FILL IN YOUR API KEY HERE"
BASE_URL = "https://api.llm.acc.unibuc.ro/v1"
MODEL_NAME = "Qwen3.8-27B"

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    api_key=API_KEY,
    base_url=BASE_URL,
)


## What should we observe?

A useful trace answers: **what happened, in what order, how long did it take, and what did it cost?** A root agent run can contain model-generation spans, tool spans, retries, and errors. Add business context with low-cardinality metadata (for example `environment="lab"` and `agent_version="v1"`), but avoid putting secrets or unrestricted prompts in tags.

Traces help debug one request. Metrics aggregate many requests: latency percentiles, error rate, token usage, tool-call frequency, and cost. Evaluations add quality signals such as answer correctness, groundedness, or whether the agent used an allowed tool. A production setup normally needs all three.

## A small agent to instrument

The deterministic tool gives us a child span that is easy to recognize. In a real service, tools would call databases or APIs and should record their status, latency, and a redacted summary of their inputs and outputs.


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_course_topic(topic: str) -> str:
    """Return a short fact about a software engineering topic."""
    facts = {
        "tracing": "Tracing records a request as a tree of nested operations.",
        "monitoring": "Monitoring aggregates operational signals and alerts on changes.",
    }
    return facts.get(topic.lower(), "No fact is available for that topic.")

agent = create_agent(model, tools=[lookup_course_topic])
question = "Use the course tool to explain tracing in one sentence."


## 1. LangSmith

[LangSmith](https://www.langchain.com/langsmith-platform) is the most integrated option for LangChain applications. Its hosted UI makes it quick to inspect run trees, prompts, tool calls, latency, token usage, and errors. The trade-off is that it is a proprietary hosted service: review its data-region, retention, access-control, and pricing policies before sending production traffic.

Create a free account using the [LangSmith administration guide](https://docs.langchain.com/langsmith/admin). Select the **EU data region** if appropriate for your team. Refuse paid features for this lab; we only need tracing.


In [ ]:
import os

os.environ["LANGSMITH_API_KEY"] = "FILL IN YOUR LANGSMITH API KEY HERE"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tsis-lab4-tracing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

agent = create_agent(model, tools=[lookup_course_topic])
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={"run_name": "course-topic-question", "tags": ["lab4", "development"]},
)
print(result["messages"][-1].content)


Open the project in LangSmith and locate `course-topic-question`. Expand the run tree and identify:

1. the root agent run;
2. the model call that decided to use the tool;
3. the tool execution and its output;
4. the final model call and total latency.

Now run the agent once with an invalid or unavailable topic. How does the trace differ? Which fields would help an on-call engineer reproduce the failure?

### Trace metadata and feedback

Names and tags make traces searchable; metadata makes them filterable. In production, pass a stable request ID and a privacy-reviewed user or session identifier rather than an email address. Feedback can come from a user rating, an evaluator, or an offline test suite.


In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={
        "run_name": "course-topic-question-v1",
        "tags": ["lab4", "agent-v1", "evaluation-candidate"],
        "metadata": {
            "environment": "notebook",
            "request_id": "demo-001",
            "agent_version": "v1",
        },
    },
)
print(result["messages"][-1].content)
# In the UI, attach a score such as correctness=1 or correctness=0 to this run.


## 2. LangFuse: an open-source alternative

[LangFuse](https://langfuse.com/) is an open-source LLM observability platform. Its hosted Cloud service is the simplest way to try it; the same product can also be self-hosted when an organization needs control over data location and retention. Self-hosting moves operational responsibility to your team: upgrades, backups, authentication, and availability are now your concerns.

Create a project at [cloud.langfuse.com](https://cloud.langfuse.com/) and copy its public and secret keys. For a self-hosted deployment, use the URL and credentials supplied by the administrator. The callback handler creates nested LangChain observations.


In [ ]:
os.environ["LANGFUSE_PUBLIC_KEY"] = "FILL IN YOUR LANGFUSE PUBLIC KEY HERE"
os.environ["LANGFUSE_SECRET_KEY"] = "FILL IN YOUR LANGFUSE SECRET KEY HERE"
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

from langfuse import get_client
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config={
        "callbacks": [langfuse_handler],
        "run_name": "course-topic-question",
        "tags": ["lab4", "development"],
        "metadata": {
            "langfuse_session_id": "lab4-demo-session",
            "langfuse_user_id": "student-demo",
        },
    },
)
print(result["messages"][-1].content)
# Short-lived programs should flush before exiting; notebooks can do this explicitly.
get_client().flush()


### Compare the two platforms

Run the same question through both integrations. Compare:

| Dimension | LangSmith | LangFuse |
| --- | --- | --- |
| Primary appeal | Fastest LangChain setup and polished hosted workflow | Open-source core and self-hosting option |
| Integration here | Environment variables and automatic LangChain tracing | `CallbackHandler` passed in invocation config |
| Operational responsibility | Mostly delegated to the vendor | Cloud or your team, depending on deployment |
| Questions to verify | Region, retention, access, export, cost | Hosting, upgrades, backups, retention, cost |

There is no meaningful observability comparison unless the recorded inputs and outputs are equivalent. Check that both traces contain the same agent, model, and tool steps, then compare filtering, latency views, token accounting, and feedback/evaluation workflows.


## 3. OpenTelemetry: portable telemetry

[OpenTelemetry](https://opentelemetry.io/) is a vendor-neutral standard and SDK for traces, metrics, and logs. An application emits spans once; an **OTel Collector** can batch, sample, redact, and fan them out to several backends. This reduces vendor lock-in, but introduces configuration and operational complexity. OTel by itself is not an LLM dashboard: the backend still needs to understand AI-specific attributes if you want token and model views.

The following is an annotated configuration sketch for sending OTLP/HTTP to LangSmith. It is intentionally not executed: replace the endpoint and credentials with your organization's approved collector or backend. The endpoint is a base URL ending in `/otel`; the exporter appends `/v1/traces`.


In [ ]:
# Annotated OTLP/HTTP configuration (do not paste production secrets in a notebook).
os.environ["OTEL_SERVICE_NAME"] = "tsis-agent"
os.environ["OTEL_RESOURCE_ATTRIBUTES"] = "deployment.environment=lab,service.version=v1"
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = "x-api-key=<approved-key>"

# A Python application would initialize a TracerProvider and OTLP exporter:
# from opentelemetry import trace
# from opentelemetry.sdk.resources import Resource
# from opentelemetry.sdk.trace import TracerProvider
# from opentelemetry.sdk.trace.export import BatchSpanProcessor
# from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
# provider = TracerProvider(resource=Resource.create({"service.name": "tsis-agent"}))
# provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
# trace.set_tracer_provider(provider)
# tracer = trace.get_tracer("tsis-lab4")
# with tracer.start_as_current_span("agent-request") as span:
#     span.set_attribute("agent.version", "v1")
#     agent.invoke(...)

print("OTel variables shown; exporter initialization intentionally left as an exercise.")


## Production checklist

Before enabling tracing for real users, decide explicitly:

- **Data minimization:** redact secrets, personal data, credentials, and full documents; capture hashes or summaries where possible.
- **Access and residency:** who can see prompts and tool outputs, in which region, for how long, and under which audit policy?
- **Reliability:** tracing must not take down the agent; use bounded queues, timeouts, sampling, and an appropriate fail-open/fail-closed policy.
- **Cost:** measure events, retained tokens, storage, and egress; sample routine traffic but retain errors and representative slow traces.
- **Signal quality:** define alerts for error rate, latency percentiles, token/cost spikes, tool failures, loops, and policy violations.
- **Evaluation:** replay a versioned test set after prompt/model/tool changes and connect quality scores to trace IDs.

### Discussion

For a regulated service with an existing OTel Collector, which design would you choose and why? For a small team debugging a new LangGraph agent, which platform would you start with? State your assumptions about data sensitivity, staff capacity, and portability.
